# 🎬 VibeFrame Quick Start - Colab Free Tier

**Simple music video generation in 5 minutes**

![VibeFrame](https://img.shields.io/badge/VibeFrame-2.0-blue)

---

## 📋 Steps

1. ⬇️ Install dependencies
2. 🔗 Mount Google Drive
3. 🎵 Set your prompts
4. ▶️ Run all cells
5. ⬇️ Download your video

---


In [ ]:
# @title ⬇️ Step 1: Install Dependencies
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import subprocess, sys
packages = ['torch', 'transformers', 'diffusers>=0.30.0', 'moviepy', 'google-colab']
for p in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])

print("✅ Dependencies installed!")

In [ ]:
# @title 🔗 Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/VibeFrame', exist_ok=True)
print("✅ Drive mounted!")

In [ ]:
# @title 🎵 Step 3: Your Music Video Prompts
# @markdown Edit these prompts for your video

prompts = [
    "A wooden toy ship sailing on blue carpet waves, cinematic lighting, smooth tracking shot",
    "Close-up of wooden ship hull with intricate carvings, warm golden lighting",
    "Young woman with guitar in bamboo forest, golden hour lighting, dreamy atmosphere",
    "Panda with red jacket playing guitar in bamboo forest, soft sunlight, close-up",
    "Wooden toy ship arriving at tiny harbor made of building blocks, children's room setting",
]

# Settings (adjust for memory)
steps = 25       # @param {type:"slider", min:10, max:50}
frames = 49      # @param {type:"slider", min:16, max:100}
height = 480     # @param [240, 360, 480, 576]
width = 720      # @param [480, 720, 1024]

print(f"✅ {len(prompts)} scenes configured")
print(f"⚙️  {steps} steps, {frames} frames, {width}x{height}")

In [ ]:
# @title ▶️ Step 4: Generate Video
import torch, gc
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
from moviepy.editor import VideoFileClip, concatenate_videoclips
from datetime import datetime
from google.colab import files

def clear_mem():
    gc.collect()
    torch.cuda.empty_cache()

output_dir = '/content/drive/MyDrive/VibeFrame'
clips = []

print("🎬 Starting generation...\n")

for i, prompt in enumerate(prompts):
    print(f"Generating clip {i+1}/{len(prompts)}: {prompt[:40]}...")
    
    pipe = CogVideoXPipeline.from_pretrained(
        "THUDM/CogVideoX-2b", torch_dtype=torch.float16, variant="fp16"
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_slicing()
    pipe.to("cuda")
    
    video = pipe(prompt, num_inference_steps=steps, num_frames=frames, 
                 height=height, width=width).frames[0]
    
    path = f"{output_dir}/clip_{i:03d}.mp4"
    export_to_video(video, path, fps=8)
    clips.append({'path': path, 'duration': len(video)/8})
    
    del pipe
    clear_mem()
    
    if (i+1) % 2 == 0:
        print(f"  🧹 Memory cleanup\n")

# Assemble
print("\n🎞️ Assembling final video...")
video_clips = [VideoFileClip(c['path']) for c in clips]
final = concatenate_videoclips(video_clips)
final_path = f"{output_dir}/vibemv_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"
final.write_videofile(final_path, fps=24, codec='libx264')

# Cleanup
for vc in video_clips:
    vc.close()
final.close()

print(f"\n✅ Done! Video saved to: {final_path}")

In [ ]:
# @title ⬇️ Step 5: Download Video
# @markdown Run this cell to download your video

from google.colab import files
import os

# Find latest video
videos = [f for f in os.listdir(output_dir) if f.endswith('.mp4') and 'vibemv_' in f]
if videos:
    latest = max(videos, key=lambda x: os.path.getmtime(f"{output_dir}/{x}"))
    print(f"📹 Downloading: {latest}")
    files.download(f"{output_dir}/{latest}")
else:
    print("❌ No video found!")

---

## 💡 Tips

| Problem | Solution |
|---------|----------|
| Out of memory | Reduce height/width or steps |
| Too slow | Reduce frames to 32 |
| Poor quality | Increase steps to 35 |
| Colab disconnects | Save to Drive often |

**Need more features?** Try the full `VibeFrame_Colab_Optimized.ipynb` notebook!